# 3. DDL i Import podataka u MySQL

Ovaj notebook:
1. Kreira bazu podataka `fipu_srp_projekt` (ako ne postoji)
2. Ispravlja tipove podataka u Pandas-u (datumi, integeri)
3. Kreira staging tablicu `support_tickets` s eksplicitno definiranim SQL tipovima
4. Importira 80% skup podataka

**Preduvjeti:** `.env` datoteka s `DB_PASSWORD`, `python-dotenv`, `pymysql`

In [1]:
import os
import pandas as pd
from sqlalchemy import create_engine, text
from sqlalchemy.types import Integer, Float, String, DateTime
from dotenv import load_dotenv

load_dotenv()

CSV_FILE_PATH = 'Support_tickets_PROCESSED.csv'

In [2]:
import os
from dotenv import load_dotenv
from sqlalchemy import create_engine, text

load_dotenv()

DB_USER = os.getenv('DB_USER', 'root')
DB_PASSWORD = os.getenv('DB_PASSWORD')
DB_HOST = os.getenv('DB_HOST', 'localhost')
DB_NAME = os.getenv('DB_NAME', 'fipu_srp_projekt')

if not DB_PASSWORD:
    raise ValueError("DB_PASSWORD nije postavljen! Kreiraj .env datoteku.")

# Engine bez baze - za CREATE DATABASE
engine_init = create_engine(f"mysql+pymysql://{DB_USER}:{DB_PASSWORD}@{DB_HOST}")
print(f"Povezano na MySQL server: {DB_HOST}")

Povezano na MySQL server: localhost


## 3.1 Kreiranje baze podataka

In [3]:
with engine_init.connect() as conn:
    conn.execute(text(f"CREATE DATABASE IF NOT EXISTS {DB_NAME}"))
    conn.commit()
    print(f"Baza podataka '{DB_NAME}' je spremna.")

Baza podataka 'fipu_srp_projekt' je spremna.


## 3.2 Učitavanje i transformacija podataka

In [4]:
df = pd.read_csv(CSV_FILE_PATH, delimiter=',')
print(f"CSV učitan: {df.shape[0]} redaka, {df.shape[1]} stupaca")

# Float -> Int za stupce bez NaN
for col in ['id', 'issue_num', 'issue_contr_count']:
    df[col] = df[col].astype(int)

# String -> DateTime
for col in ['started', 'ended', 'issue_created', 'issue_resolution_date', 'last_change_date']:
    df[col] = pd.to_datetime(df[col], format='ISO8601', utc=True)

print(f"\nTipovi nakon konverzije:")
print(df.dtypes)

CSV učitan: 53353 redaka, 42 stupaca

Tipovi nakon konverzije:
id                                             int64
started                          datetime64[ns, UTC]
ended                            datetime64[ns, UTC]
issue_num                                      int64
issue_proj                                    object
issue_reporter                                object
issue_assignee                                object
issue_contr_count                              int64
issue_type                                    object
issue_priority                                object
issue_created                    datetime64[ns, UTC]
issue_resolution_date            datetime64[ns, UTC]
issue_resolution                              object
issue_status                                  object
issue_comments_count                           int64
last_change_date                 datetime64[ns, UTC]
wfe_in_review                                  int64
wfe_deployment                      

## 3.3 Definicija SQL tipova

In [5]:
sql_dtypes = {
    # Identifikatori
    'id':                             Integer,
    'issue_num':                      Integer,
    # Datumi -> DATETIME (umjesto TEXT)
    'started':                        DateTime,
    'ended':                          DateTime,
    'issue_created':                  DateTime,
    'issue_resolution_date':          DateTime,
    'last_change_date':               DateTime,
    # Stringovi -> VARCHAR (umjesto TEXT)
    'issue_proj':                     String(255),
    'issue_reporter':                 String(255),
    'issue_assignee':                 String(255),
    'issue_type':                     String(50),
    'issue_priority':                 String(50),
    'issue_resolution':               String(50),
    'issue_status':                   String(50),
    # Numerički
    'issue_contr_count':              Integer,
    'issue_comments_count':           Integer,
    'processing_steps':               Integer,
    # Workflow entry counts (wfe_) -> INT
    'wfe_in_review': Integer, 'wfe_deployment': Integer,
    'wfe_resolved': Integer, 'wfe_open': Integer,
    'wfe_monitoring': Integer, 'wfe_done': Integer,
    'wfe_pending_customer_approval': Integer, 'wfe_rejected': Integer,
    'wfe_testing_monitoring': Integer, 'wfe_in_progress': Integer,
    'wfe_reopened': Integer, 'wfe_to_do': Integer,
    'wfe_validation': Integer, 'wfe_resolved_under_monitoring': Integer,
    'wfe_closed': Integer, 'wfe_waiting': Integer,
    'wfe_cancelled': Integer, 'wfe_under_review': Integer,
    'wfe_approved': Integer, 'wfe_pending_deployment': Integer,
    # Workflow time (wf_) -> FLOAT (sekunde s decimalama)
    'wf_resolved': Float, 'wf_open': Float,
    'wf_in_progress': Float, 'wf_waiting': Float,
    'wf_total_time': Float,
}
print(f"Definirano {len(sql_dtypes)} eksplicitnih SQL tipova.")

Definirano 42 eksplicitnih SQL tipova.


## 3.4 Import u MySQL

In [6]:
engine = create_engine(f"mysql+pymysql://{DB_USER}:{DB_PASSWORD}@{DB_HOST}/{DB_NAME}")

df.to_sql(
    name='support_tickets',
    con=engine,
    if_exists='replace',
    index=False,
    dtype=sql_dtypes,
    chunksize=5000
)
print(f"Tablica 'support_tickets' kreirana i {len(df)} redaka uvezeno.")

Tablica 'support_tickets' kreirana i 53353 redaka uvezeno.


## 3.5 Verifikacija SQL tipova

In [7]:
with engine.connect() as conn:
    result = conn.execute(text("DESCRIBE support_tickets"))
    rows = result.fetchall()

print(f"{'Stupac':<40} {'SQL tip':<20}")
print("-" * 60)
for row in rows:
    print(f"{row[0]:<40} {row[1]:<20}")

Stupac                                   SQL tip             
------------------------------------------------------------
id                                       int                 
started                                  datetime            
ended                                    datetime            
issue_num                                int                 
issue_proj                               varchar(255)        
issue_reporter                           varchar(255)        
issue_assignee                           varchar(255)        
issue_contr_count                        int                 
issue_type                               varchar(50)         
issue_priority                           varchar(50)         
issue_created                            datetime            
issue_resolution_date                    datetime            
issue_resolution                         varchar(50)         
issue_status                             varchar(50)         
issue_com